Vectorless Rag - Page index

In [1]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [9]:
from pageindex import PageIndexClient
from groq import Client
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Client(api_key=GROQ_API_KEY)
groq_client

uploading doc to pageindex

In [3]:
pdf_path = "./sample_doc.pdf"
result = pi_client.submit_document(pdf_path)
doc_id = result["doc_id"]
print("Uploaded")
doc_id


Uploaded


'pi-cmruwnls2006b01o20e136li5'

building the index tree

In [4]:
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")
    if status == "completed":
        print("Tree index ready")
        break
    elif status == "failed":
        print("Processing failed")
        break
    time.sleep(5)

Status: completed
Tree index ready


In [5]:
tree_result = pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree = tree_result.get("result",[])
print(f"Top level sections: {len(pageindex_tree)}")
print("\n Tree first node: ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top level sections: 1

 Tree first node: 
{
  "title": "Unit 5 \u2013 Reliable Group Communication & Naming Systems",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Unit 5 \u2013 Reliable Group Communication & Naming Systems\n\nThese notes explain every topic and subtopic from the uploaded Unit 5 PDF in a simpler and more student-friendly way. Examples are included throughout so that the concepts are easier to understand and remember for exams.\n",
  "text": "# Unit 5 \u2013 Reliable Group Communication & Naming Systems\n\nThese notes explain every topic and subtopic from the uploaded Unit 5 PDF in a simpler and more student-friendly way. Examples are included throughout so that the concepts are easier to understand and remember for exams.\n",
  "nodes": [
    {
      "title": "Reliable Group Communication",
      "node_id": "0001",
      "page_index": 1,
      "summary": "The text defines reliable multicast as a method for ensuring consistent message delivery to multip

Pretty printing the whole tree

In [6]:
def print_tree(nodes,indent=0):
    """Recursively print tree for a visual overview"""
    for node in nodes:
        prefix = " " * indent + ("-" if indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)
        
print("Full doc structure")
print_tree(pageindex_tree)

Full doc structure
[0000] Unit 5 – Reliable Group Communication & Naming Systems (p.1)
 -[0001] Reliable Group Communication (p.1)
 -[0002] Scalability in Reliable Multicasting (p.1)
 -[0003] Non-Hierarchical Feedback Control (SRM) (p.2)
 -[0004] Hierarchical Feedback Control (p.2)
 -[0005] Atomic Multicast (p.3)
 -[0006] Virtual Synchrony (p.3)
 -[0007] Message Ordering (p.4)
 -[0008] Address (p.6)
 -[0009] Identifiers (p.6)
 -[0010] Name Resolution (p.6)
 -[0011] Flat Naming (p.7)
 -[0012] Broadcasting (p.7)
 -[0013] Multicasting (p.7)
 -[0014] Forwarding Pointers (p.8)
 -[0015] Home-Based Approaches (p.8)
 -[0016] Hierarchical Approaches (p.9)
 -[0017] Structured Naming (p.9)
 -[0018] Name Resolution in Structured Naming (p.9)
 -[0019] Linking and Mounting (p.10)
 -[0020] Implementation of Name Space (p.10)
 -[0021] Implementation of Name Resolution (p.11)
 -[0022] Case Study: DNS (p.11)
 -[0023] Attribute-Based Naming (p.12)


In [7]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total Nodes in tree: {total}")

Total Nodes in tree: 24


LLM Tree Search

In [15]:
def llm_tree_search(query: str,tree: list,model: str = "llama-3.1-8b-instant") -> dict:
    """
    Core Page Index retrieval
    """
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index","?"),
                "summary": n.get("text","")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document tree structure (like a Table of contents).
    Your task: identify which nodes IDS most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.
    
    Query: {query}
    
    Document Tree:
    {json.dumps(compressed_tree,indent=2)}

    Reply only in this exact JSON format:
    {{
      "thinking": "<your step-by-step reasoning>",
      "node_list": ["node-id1","node_id2"]
    }}
    """
    response = groq_client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    response_format={"type":"json_object"}
    )

    return json.loads(response.choices[0].message.content)



testing 

In [16]:
query = "What is covered in Reliable Group Communication?"
print(f"Query: {query}")
result = llm_tree_search(query,pageindex_tree)
print("LLM Reasoning:")
print(result.get("thinking","N/A"))
print("Selected Node IDs: ",result.get("node_list",[]))


Query: What is covered in Reliable Group Communication?
LLM Reasoning:
Given the query 'What is covered in Reliable Group Communication?', the first step is to find the top-level node that is most relevant to the query. The query keyword 'Reliable' is present in the title of node '0001', which is 'Reliable Group Communication'. Therefore, this node is likely to contain the answer to the query.
Selected Node IDs:  ['0001']
